## Importação de bibliotecas

In [2]:
!pip install python-dotenv -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes
!pip install transformers trl datasets
!pip install sentence-transformers pypdf -q
!pip install spacy -q
!python -m spacy download pt_core_news_lg -q

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-gsnfcvz8/unsloth_8677c329c87540c1b95bb93f647db318
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-gsnfcvz8/unsloth_8677c329c87540c1b95bb93f647db318
  Resolved https://github.com/unslothai/unsloth.git to commit de1f4a743409f0c8403947d6a94144b65dc67635
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.2 MB/s eta 0:00:00
   ━━━━

As bibliotecas abaixo cobrem quatro partes do notebook: chamadas à API da OpenAI (`requests`, `openai`), manipulação de dados (`pandas`, `json`), anonimização (`re`, `spacy`) e fine-tuning do modelo (`unsloth`, `torch`, `datasets`, `trl`, `transformers`).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import requests
import json
import os
import pandas as pd
import re
import spacy
from openai import OpenAI
from dotenv import load_dotenv
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Carrega a chave da API armazenada em `api-key.txt`, inicializa o cliente da OpenAI e carrega o modelo de linguagem do spaCy em português, usado mais adiante na etapa de anonimização.

In [5]:
load_dotenv('/content/drive/MyDrive/Tech_Challenge_3/api-key.txt')
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
nlp = spacy.load("pt_core_news_lg")

Baixa o dataset público `healthqa-br` (perguntas e respostas clínicas em português) e seleciona apenas as colunas usadas na etapa seguinte. Essa é a primeira das duas fontes de dados do fine-tuning.

In [6]:
dados_dataset = pd.read_parquet("hf://datasets/Larxel/healthqa-br/healthqa-br.parquet")


In [7]:
qxa = dados_dataset[["question", "answer"]]
qxa.head()


,question,answer
0,"Homem com 49 anos de idade apresenta, há um an...",C
1,Mulher com 72 anos de idade vem fazendo tratam...,A
2,Homem com 26 anos de idade procura atendimento...,E
3,Menina com 12 anos de idade tem diagnóstico de...,E
4,"Mulher com 19 anos de idade, primigesta, com g...",B


## Pré-processamento

Para cada par pergunta/resposta do dataset público, `summarize_dataset` chama o LLM da OpenAI para extrair sintoma, diagnóstico e tratamento em formato JSON. Respostas vazias ou com JSON inválido são registradas no console e a linha é pulada, em vez de interromper o processo inteiro. Os resultados parciais são salvos a cada 50 itens, para não perder tudo se o processo cair no meio do caminho.

Carrega os laudos internos (simulados) a partir de uma planilha Excel e seleciona as colunas usadas no fine-tuning: o laudo estruturado, o diagnóstico e o tratamento sugerido.

In [ ]:
def summarize_dataset(dataset):
    symptom_x_diagnosis = []

    for content in dataset.itertuples():
        try:
            response = client.chat.completions.create(
                model="gpt-5.6-luna", # Modelo com menor custo para requisições em massa
                response_format={"type": "json_object"},
                reasoning_effort="low", # Indica o quanto o modelo vai "pensar" para resolver a tarefa. Valores menores custam menos e são mais rápidas
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Analise o conteúdo a seguir e identifique qual é o problema de saúde principal envolvido. "
                            "Responda em JSON com os campos: 'symptom' (o sintoma ou quadro clínico descrito), "
                            "'diagnosis' (o diagnóstico correto, com base na alternativa certa) e 'treatment' "
                            "(o tratamento para o diagnóstico, se tiver. Se não tiver, escreva 'Sem tratamento definido'). "
                            "Seja objetivo: cada campo deve ter no máximo uma frase curta, sem explicações adicionais."
                        )
                    },
                    {
                        "role": "user",
                        "content": f"{content.question} = {content.answer}\n###"
                    }
                ],
                max_completion_tokens=900,
            )

            result_text = response.choices[0].message.content.strip()

            if not result_text:
                print(f"Linha {content.Index}: resposta vazia (finish_reason={response.choices[0].finish_reason}). Pulando.")
                continue

            result = json.loads(result_text)

        # Tratando linhas com erros na geração do JSON
        except json.JSONDecodeError as e:
            print(f"Linha {content.Index}: JSON inválido ({e}). Pulando.")
            continue
        except Exception as e:
            print(f"Linha {content.Index}: erro na chamada ({e}). Pulando.")
            continue

        symptom_x_diagnosis.append({
            "symptom": result.get("symptom"),
            "diagnosis": result.get("diagnosis"),
            "treatment": result.get("treatment"),
        })

        # Salvando os resultados no arquivo JSON parcialmente, para evitar que um erro faça todos os dados serem perdidos
        if len(symptom_x_diagnosis) % 50 == 0:
            with open('symptom_x_diagnosis.json', 'w') as json_file:
                json.dump({"symptom_x_diagnosis": symptom_x_diagnosis}, json_file)

    with open('symptom_x_diagnosis.json', 'w') as json_file:
        json.dump({"symptom_x_diagnosis": symptom_x_diagnosis}, json_file)

summarize_dataset(qxa)

In [8]:
df = pd.read_excel("/content/drive/MyDrive/Tech_Challenge_3/laudos-fraturas.xlsx")
excel_dataset = df[["Laudo estruturado", "Diagnóstico", "Tratamento/conduta sugerida"]]
excel_dataset.head()


,Laudo estruturado,Diagnóstico,Tratamento/conduta sugerida
0,"Campos pulmonares bem expandidos, sem opacidad...",Normal,Sem tratamento específico. Seguimento conforme...
1,"Fratura transversal do rádio distal, com discr...",Fratura distal do rádio,Imobilização e avaliação ortopédica. Redução s...
2,Alinhamento articular preservado. Interlinhas ...,Normal,Sem tratamento específico. Analgesia se houver...
3,"Traço de fratura oblíquo no maléolo lateral, s...",Fratura do maléolo lateral,"Imobilização, restrição de carga e avaliação o..."
4,Corpos vertebrais com alturas preservadas. Ali...,Normal,"Tratamento conservador se houver dor, conforme..."


## Anonimização

Funções utilizadas para limpar datasets antes do treinamento, no intuito de preservar a identidade dos pacientes. Está sendo utilizado apenas no Excel pois o dataset healthqa já está limpo, porém o ideal é passar essas funções antes de qualquer dado sensível ser usado para treinamento.

Duas funções, cada uma cobrindo um tipo de identificador diferente:

- `anonimize_text` usa expressões regulares para mascarar padrões previsíveis (CPF, telefone, data, número de prontuário).
- `remove_names` usa o modelo de NER do spaCy para encontrar e substituir nomes de pessoas, locais e organizações citados em texto livre (algo que regex sozinho não pega, como "Paciente João da Silva...").

In [9]:
def anonimize_text(text):
    text = re.sub(r'\d{3}\.?\d{3}\.?\d{3}-?\d{2}', '[CPF]', text)
    text = re.sub(r'\(\d{2}\)\s?\d{4,5}-?\d{4}', '[TELEFONE]', text)
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', '[DATA]', text)
    text = re.sub(r'(prontuário|registro)\s*n?[ºo°]?\s*\d+', '[PRONTUARIO]', text, flags=re.I)
    return text

def remove_names(text):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ("PER", "LOC", "ORG"):
            text = text.replace(ent.text, f"[{ent.label_}]")
    return text


## Curadoria

Antes de consolidar o dataset final, aplicamos duas checagens simples sobre os registros já anonimizados:

- `remove_duplicates` descarta registros com o mesmo sintoma repetido (mantém só a primeira ocorrência).
- `filter_incomplete` descarta registros em que sintoma, diagnóstico ou tratamento vieram vazios, o que pode acontecer tanto na geração via LLM quanto na planilha de origem.

São funções propositalmente simples: não fazem deduplicação semântica nem revisão por LLM, só uma limpeza básica para não deixar lixo óbvio entrar no fine-tuning.

In [11]:
def remove_duplicates(data):
    seen = set()
    unique_data = []
    for item in data:
        key = (item.get("symptom") or "").strip().lower()
        if key and key not in seen:
            seen.add(key)
            unique_data.append(item)
    return unique_data

def filter_incomplete(data):
    return [
        item for item in data
        if item.get("symptom") and item.get("diagnosis") and item.get("treatment")
    ]


## Processamento do dataset sintético (Excel)

`process_excel_dataset` junta os dados já gerados pelo LLM (arquivo `symptoms_x_diagnosis.json`) com os laudos do Excel. Estes últimos anonimizados na hora de entrar na lista. Em seguida aplicamos as duas funções de curadoria definidas acima e só então salvamos o resultado final em `diagnosis_dataset_chat_data.json`, que é o arquivo usado no fine-tuning.

In [12]:
def process_excel_dataset(dataset, file_path, processed_data):
    with open(file_path, 'r', encoding='utf-8') as file:
        json_data = json.load(file)
        qxa_list = json_data["symptom_x_diagnosis"]

        for _, item in dataset.iterrows():
          qxa_list.append({
              "symptom": anonimize_text(remove_names(item["Laudo estruturado"])),
              "diagnosis": item["Diagnóstico"],
              "treatment": anonimize_text(item["Tratamento/conduta sugerida"]),
          })
        processed_data.extend(qxa_list)

processed_data = []

# Adicionar dados do excel no arquivo
process_excel_dataset(excel_dataset, '/content/drive/MyDrive/Tech_Challenge_3/symptom_x_diagnosis.json', processed_data)

# Remove duplicatas e registros incompletos antes de salvar
total_antes = len(processed_data)
processed_data = remove_duplicates(processed_data)
processed_data = filter_incomplete(processed_data)
print(f"Curadoria: {total_antes - len(processed_data)} registro(s) removido(s) ({total_antes} -> {len(processed_data)}).")

# Salvar todos os dados processados em um arquivo JSON
output_filename = r'diagnosis_dataset_chat_data.json'
with open(output_filename, 'w', encoding='utf-8') as file:
    json.dump(processed_data, file, ensure_ascii=False, indent=4)

print(f"Todos os dados reformatados foram salvos em '{output_filename}'.")


Curadoria: 234 registro(s) removido(s) (5682 -> 5448).
Todos os dados reformatados foram salvos em 'diagnosis_dataset_chat_data.json'.


## Protocolos internos do hospital (contexto para o treino)

In [13]:
from pypdf import PdfReader

PROTOCOLOS_DIR = "/content/drive/MyDrive/Tech_Challenge_3/protocolos"

def carregar_texto_pdf(caminho):
    leitor = PdfReader(caminho)
    return "\n".join((pagina.extract_text() or "") for pagina in leitor.pages)

def dividir_em_chunks(texto, tamanho=500, sobreposicao=80):
    chunks = []
    inicio = 0
    while inicio < len(texto):
        trecho = texto[inicio:inicio + tamanho].strip()
        if trecho:
            chunks.append(trecho)
        inicio += tamanho - sobreposicao
    return chunks

protocolo_chunks = []

pdfs_encontrados = (
    os.path.isdir(PROTOCOLOS_DIR)
    and any(f.lower().endswith(".pdf") for f in os.listdir(PROTOCOLOS_DIR))
)

if pdfs_encontrados:
    for nome_arquivo in sorted(os.listdir(PROTOCOLOS_DIR)):
        if not nome_arquivo.lower().endswith(".pdf"):
            continue
        caminho = os.path.join(PROTOCOLOS_DIR, nome_arquivo)
        texto_completo = carregar_texto_pdf(caminho)
        for chunk in dividir_em_chunks(texto_completo):
            protocolo_chunks.append({"texto": chunk, "fonte": nome_arquivo})
    print(f"{len(protocolo_chunks)} trechos carregados de PDFs em {PROTOCOLOS_DIR}")
else:
    print(f"Nenhum PDF encontrado em {PROTOCOLOS_DIR} -- usando protocolos de exemplo (fallback).")
    protocolos_texto = {
        "protocolo_fraturas_v3.pdf": (
            "Protocolo de fratura de femur: priorizar reducao cirurgica em ate 48h em idosos, "
            "com avaliacao de risco cardiovascular pre-operatorio."
        ),
        "protocolo_endocrino_v2.pdf": (
            "Protocolo de diabetes tipo 2: revisar hemograma e funcao renal antes de ajustar dose "
            "de metformina em pacientes acima de 65 anos."
        ),
        "politica_ia_hospital.pdf": (
            "Politica geral do hospital: qualquer sugestao de tratamento gerada por IA deve ser "
            "validada por um medico responsavel antes de ser aplicada ao paciente."
        ),
    }
    for fonte, texto in protocolos_texto.items():
        for chunk in dividir_em_chunks(texto):
            protocolo_chunks.append({"texto": chunk, "fonte": fonte})

1142 trechos carregados de PDFs em /content/drive/MyDrive/Tech_Challenge_3/protocolos


Para cada exemplo do dataset consolidado, buscamos por embeddings o trecho de protocolo mais parecido com o sintoma e anexamos como `protocol_context` quando a similaridade passa de um limiar (`0.35`, ajustável). Quando nenhum trecho é parecido o suficiente, o exemplo fica sem contexto de protocolo, como estava antes — o modelo precisa aprender os dois casos: com e sem protocolo disponível, já que nem todo sintoma tem um protocolo interno relevante.

In [14]:
from sentence_transformers import SentenceTransformer, util

LIMIAR_SIMILARIDADE = 0.35

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

textos_protocolo = [p["texto"] for p in protocolo_chunks]
embeddings_protocolo = embedder.encode(textos_protocolo, show_progress_bar=False) if textos_protocolo else []

textos_sintoma = [item["symptom"] for item in processed_data]
embeddings_sintoma = embedder.encode(textos_sintoma, show_progress_bar=False)

qtd_com_protocolo = 0
for item, emb_sintoma in zip(processed_data, embeddings_sintoma):
    item["protocol_context"] = None
    if len(embeddings_protocolo) == 0:
        continue
    similaridades = util.cos_sim(emb_sintoma, embeddings_protocolo)[0]
    melhor_idx = int(similaridades.argmax())
    if float(similaridades[melhor_idx]) >= LIMIAR_SIMILARIDADE:
        item["protocol_context"] = protocolo_chunks[melhor_idx]["texto"]
        qtd_com_protocolo += 1

print(f"{qtd_com_protocolo}/{len(processed_data)} exemplos receberam contexto de protocolo interno.")

# Resalva o dataset consolidado, agora com o campo protocol_context incluso
with open("diagnosis_dataset_chat_data.json", "w", encoding="utf-8") as file:
    json.dump(processed_data, file, ensure_ascii=False, indent=4)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

5355/5448 exemplos receberam contexto de protocolo interno.


## Configuração do Unsloth para treinamento do modelo

Define os caminhos de entrada/saída do dataset formatado e os parâmetros de quantização em 4-bit, usados para carregar o modelo consumindo bem menos memória de GPU. A lista `fourbit_models` é só referência de modelos compatíveis com o Unsloth — o modelo escolhido para este notebook é o `llama-3-8b-bnb-4bit`.

In [15]:
DATA_PATH = "/content/drive/MyDrive/Tech_Challenge_3/diagnosis_dataset_chat_data.json"
OUTPUT_PATH_DATASET = "/content/drive/MyDrive/Tech_Challenge_3/formatted_diagnosis_dataset_chat_data.json"
max_seq_length = 2048
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
]


Converte o dataset consolidado (`symptom`/`diagnosis`/`treatment`) para o formato instruction/input/output esperado pelo fine-tuning, e salva o resultado em um novo arquivo JSON.

In [16]:
INSTRUCAO_DIAGNOSTICO = (
    "Faça o diagnóstico do problema de saúde do paciente com base nos sintomas e no protocolo interno "
    "fornecido (se houver), e sugira o tratamento recomendado. Nunca prescreva diretamente -- apenas "
    "sugira condutas para validação médica."
)

def montar_input_treino(symptom, protocol_context):
    entrada = f"Pergunta do medico: {symptom}"
    if protocol_context:
        entrada += f"\nProtocolo interno relevante: {protocol_context}"
    return entrada

def format_dataset_into_model_input(data):
    instructions = []
    inputs = []
    outputs = []

    for prompt in data['train']:
        instructions.append(INSTRUCAO_DIAGNOSTICO)
        inputs.append(montar_input_treino(prompt['symptom'], prompt.get('protocol_context')))
        outputs.append(prompt['diagnosis'] + ' ' + prompt['treatment'])

    # Criando o dicionário final
    formatted_data = {
        "instruction": instructions,
        "input": inputs,
        "output": outputs
    }

    # Salvando o resultado em um arquivo JSON
    with open(OUTPUT_PATH_DATASET, 'w') as output_file:
        json.dump(formatted_data, output_file, indent=4)

    print(f"Dataset salvo em {OUTPUT_PATH_DATASET}")


Roda a conversão do dataset e carrega o modelo base (Llama-3-8B em 4-bit) junto com o tokenizer, usando o Unsloth.

In [17]:
data = load_dataset("json", data_files=DATA_PATH)
format_dataset_into_model_input(data)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset salvo em /content/drive/MyDrive/Tech_Challenge_3/formatted_diagnosis_dataset_chat_data.json
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Aplica LoRA ao modelo carregado: só uma fração pequena dos parâmetros (`r=16`) é efetivamente treinada, o que reduz bastante o custo do fine-tuning comparado a treinar o modelo inteiro.

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 47,
    use_rslora = False,
    loftq_config = None,
)


Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## Treinamento do modelo

Define o template de prompt (formato Alpaca) e formata cada exemplo do dataset final juntando instrução, input e output em um único texto, encerrado pelo token de fim de sequência (`EOS_TOKEN`).

In [19]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):

        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset

OUTPUT_PATH_DATASET = "/content/drive/MyDrive/Tech_Challenge_3/formatted_diagnosis_dataset_chat_data.json"

dataset = load_dataset("json", data_files=OUTPUT_PATH_DATASET, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5448 [00:00<?, ? examples/s]

Configura o `SFTTrainer`, que orquestra o loop de treinamento: tamanho de batch, taxa de aprendizado, número de passos e otimizador. `max_steps=60` é propositalmente baixo — ajuste para mais passos/épocas conforme o tamanho real do dataset final.

In [20]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 47,
        output_dir = "outputs",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5448 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## Validação

Inicia o treinamento propriamente dito. As perdas (`loss`) de cada passo aparecem no log.

In [21]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,448 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.785624
2,2.676228
3,2.735841
4,2.491706
5,2.203604
6,2.017079
7,1.559846
8,1.250940
9,1.005506
10,1.030286


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Testa o modelo já treinado com um exemplo de sintoma, gerando a resposta completa de uma só vez (sem exibir token a token).

In [22]:
# Input já existente no dataset. Mesma instrução e mesmo formato de input
# usados no input real pelo assistente LangGraph
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        INSTRUCAO_DIAGNOSTICO,
        montar_input_treino("Gestante de 12 semanas, assintomática, com fatores de risco para diabetes gestacional (idade materna de 35 anos, sobrepeso e antecedente de recém-nascido macrossômico), apresentando glicemia de jejum normal de 82 mg/dL.", None),
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nFaça o diagnóstico do problema de saúde do paciente com base nos sintomas e no protocolo interno fornecido (se houver), e sugira o tratamento recomendado. Nunca prescreva diretamente -- apenas sugira condutas para validação médica.\n\n### Input:\nPergunta do medico: Gestante de 12 semanas, assintomática, com fatores de risco para diabetes gestacional (idade materna de 35 anos, sobrepeso e antecedente de recém-nascido macrossômico), apresentando glicemia de jejum normal de 82 mg/dL.\n\n### Response:\nGestação de risco para diabetes gestacional. Avaliação periódica da glicemia de jejum, com monitorização do peso e do crescimento fetal.<|end_of_text|>']

In [23]:
# Segundo teste, desta vez COM um trecho de protocolo no input, para
# validar que o modelo consegue de fato condicionar a resposta nele.
protocolo_exemplo = protocolo_chunks[0]["texto"] if protocolo_chunks else None

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        INSTRUCAO_DIAGNOSTICO,
        montar_input_treino("Gestante de 12 semanas, assintomática, com fatores de risco para diabetes gestacional (idade materna de 35 anos, sobrepeso e antecedente de recém-nascido macrossômico), apresentando glicemia de jejum normal de 82 mg/dL.", protocolo_exemplo),
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)


<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Faça o diagnóstico do problema de saúde do paciente com base nos sintomas e no protocolo interno fornecido (se houver), e sugira o tratamento recomendado. Nunca prescreva diretamente -- apenas sugira condutas para validação médica.

### Input:
Pergunta do medico: Gestante de 12 semanas, assintomática, com fatores de risco para diabetes gestacional (idade materna de 35 anos, sobrepeso e antecedente de recém-nascido macrossômico), apresentando glicemia de jejum normal de 82 mg/dL.
Protocolo interno relevante: 1 
 
     
      MINISTÉRIO DA SAÚDE 
SECRETARIA DE CIÊNCIA, TECNOLOGIA E INOVAÇÃO EM SAÚDE 
 
PORTARIA SCTIE/MS Nº 13, DE 21 DE FEVEREIRO DE 2026 
 
Torna pública a decisão de atualizar, no âmbito 
do Sistema Único de Saúde - SUS, o Protocolo 
Clínico e Diretrizes Terapêuticas do Diabete 
Mel

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Diabetes gestacional, com glicemia de jejum normal (82 mg/dL). Sem tratamento definido<|end_of_text|>


Salva o adaptador LoRA e o tokenizer treinados em disco, para serem reaproveitados depois.

In [24]:
model.save_pretrained("/content/drive/MyDrive/Tech_Challenge_3/lora_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Tech_Challenge_3/lora_model")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Tech_Challenge_3/lora_model/tokenizer_config.json.


('/content/drive/MyDrive/Tech_Challenge_3/lora_model/tokenizer_config.json',
 '/content/drive/MyDrive/Tech_Challenge_3/lora_model/tokenizer.json')